In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

In [2]:
training_data = datasets.CIFAR10(
    root="data",
    train=True,
    download=True,
    transform=ToTensor(),
)

test_data = datasets.CIFAR10(
    root="data",
    train=False,
    download=True,
    transform=ToTensor(),
)

100.0%
c:\Users\Krzychu\Desktop\Informatyka\SN_PROJEKT\.venv\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [ ]:
batch_size = 16

# Create data loaders.
train_dataloader = DataLoader(training_data, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

for X, y in test_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

Shape of X [N, C, H, W]: torch.Size([64, 3, 32, 32])
Shape of y: torch.Size([64]) torch.int64


In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device} device")

Using cuda device


In [ ]:
# Define model
class ConvNet(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            # Blok 1
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.2), # Wyłącza 20% neuronów

            # Blok 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.3), # Wyłącza 30% neuronów

            # Blok 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.4), # Wyłącza 40% neuronów
        )
        
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.5), # Agresywny dropout przed klasyfikacją (50%)
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

model10 = ConvNet(num_classes=10).to(device)
model100 = ConvNet(num_classes=100).to(device)
print(model10)
print()
print(model100)

ConvNet(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU()
    (7): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU()
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU()
    (12): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU()
    (14): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=2048, out_features=512, bias=True)
    (2): ReLU()
    (3)

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer10 = torch.optim.NAdam(model10.parameters(), lr=0.001)
# Adadelta 
# Adafactor 
# Adagrad 
# Adam 
# AdamW Implements AdamW algorithm, where weight decay does not accumulate in the momentum nor variance.
# SparseAdam SparseAdam implements a masked version of the Adam algorithm suitable for sparse gradients.
# Adamax Implements Adamax algorithm (a variant of Adam based on infinity norm).
# ASGD Implements Averaged Stochastic Gradient Descent.     skoro SGD działa kiepsko to nie próbowałbym tego
# LBFGS 
# Muon 
# NAdam 
# RAdam 
# RMSprop 
# Rprop 
# SGD stochastic gradient descent                           działa kiepsko na tych

NameError: name 'model10' is not defined

In [12]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss_value, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss_value:>7f}  [{current:>5d}/{size:>5d}]")

In [ ]:
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    accuracy = 100 * correct
    print(f"Test Error: \n Accuracy: {accuracy:>0.1f}%, Avg loss: {test_loss:>8f} \n")
    return accuracy, test_loss

In [ ]:
epochs = 77
training_log = []

for t in range(epochs):
    print(f"Epoch {t+1} (CIFAR-10)\n-------------------------------")
    train(train_dataloader, model10, loss_fn, optimizer10)
    accuracy, avg_loss = test(test_dataloader, model10, loss_fn)
    training_log.append({
        "epoch": t + 1,
        "accuracy": accuracy,
        "avg_loss": avg_loss,
    })

print("Done CIFAR-10")

Epoch 1 (CIFAR-10)
-------------------------------
loss: 1.749351  [   64/50000]
loss: 1.375542  [ 6464/50000]
loss: 1.613261  [12864/50000]
loss: 1.857602  [19264/50000]
loss: 1.842233  [25664/50000]
loss: 1.484193  [32064/50000]
loss: 1.703162  [38464/50000]
loss: 1.632189  [44864/50000]
Test Error: 
 Accuracy: 39.4%, Avg loss: 1.724182 

Epoch 2 (CIFAR-10)
-------------------------------
loss: 1.896282  [   64/50000]
loss: 1.435986  [ 6464/50000]
loss: 1.751287  [12864/50000]
loss: 1.375265  [19264/50000]
loss: 1.384551  [25664/50000]
loss: 1.409546  [32064/50000]
loss: 1.608653  [38464/50000]
loss: 1.584551  [44864/50000]
Test Error: 
 Accuracy: 36.2%, Avg loss: 1.849910 

Epoch 3 (CIFAR-10)
-------------------------------
loss: 1.792582  [   64/50000]
loss: 1.495690  [ 6464/50000]
loss: 1.595021  [12864/50000]
loss: 1.541308  [19264/50000]
loss: 1.207472  [25664/50000]
loss: 1.635414  [32064/50000]
loss: 1.365386  [38464/50000]
loss: 1.261136  [44864/50000]
Test Error: 
 Accuracy:

In [ ]:
import json
import os

models_dir = "models10"
os.makedirs(models_dir, exist_ok=True)

# sprawdzamy ile jest plików w folderze
n = len([name for name in os.listdir(models_dir) if os.path.isfile(os.path.join(models_dir, name))])

# jeśli jest n modeli, to nowy będzie cifar10_(n+1)
model_path = os.path.join(models_dir, f"cifar10_{n+1}.pth")
torch.save(model10.state_dict(), model_path)

training_metadata = {
    "network_architecture": str(model10),
    "optimizer": optimizer10.__class__.__name__,
    "optimizer_defaults": optimizer10.defaults,
    "learning_rate": optimizer10.param_groups[0].get("lr"),
    "batch_size": batch_size,
    "training_log": training_log,
}

metadata_path = os.path.join(models_dir, f"{file_index}.txt")
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(training_metadata, f, ensure_ascii=False, indent=2)

print(f"Found {n} files in {models_dir}.")
print(f"Saved model to {model_path}")
print(f"Saved training metadata to {metadata_path}")

NameError: name 'model10' is not defined